In [1]:
!pip install groq python-dotenv numpy tqdm datasets

In [1]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any

load_dotenv()
random.seed(0)

client = Groq()
gsm8k_dataset = load_dataset("gsm8k", "main")

gsm8k_train = gsm8k_dataset["train"]
gsm8k_test  = gsm8k_dataset["test"]

In [2]:
def generate_response_using_Llama(
        prompt: str,
        model: str = "llama-3.1-8b-instant"
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.0, ### 수정해도 됩니다!
            stream=False
        )
        return chat_completion.choices[0].message.content
    
    except Exception as e:
        print(f"API call error: {str(e)}")
        return None

#### 응답 잘 나오는지 확인해보기

In [3]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello world! I'm here to help with any math problems you might have. What's on your mind? Do you have a specific problem you'd like me to solve, or would you like some help with a particular math concept?


#### GSM8K 데이터셋 확인해보기

In [4]:
print("[Question]")
for l in gsm8k_test['question'][0].split("."):
    print(l)
print("="*100)
print("[Answer]")
print(gsm8k_test['answer'][0])

[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


#### Util 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [5]:
### 수정해도 됩니다!
def extract_final_answer(response: str):
    regex = r"(?:Answer:|Model response:)\s*\$?([0-9,]+)\b|([0-9,]+)\s*(meters|cups|miles|minutes)"
    matches = re.finditer(regex, response, re.MULTILINE)
    results = [match.group(1) if match.group(1) else match.group(2).replace(",", "") for match in matches]

    if len(results) == 0:
        additional_regex = r"\$?([0-9,]+)"
        additional_matches = re.finditer(additional_regex, response, re.MULTILINE)
        results.extend([match.group(1).replace(",", "") for match in additional_matches])

    return results[-1] if results else None

In [6]:
### 수정해도 됩니다!
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama-3.1-8b-instant",
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        correct_answer = float(re.findall(r'\d+(?:\.\d+)?', dataset[i]["answer"].split('####')[-1])[0])

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        if response:
            if VERBOSE:
                print("="*50)
                print(response)
                print("="*50)
            predicted_answer = extract_final_answer(response)

            if predicted_answer and predicted_answer != '':
                predicted_answer = float(str(predicted_answer).replace(",", ""))
                diff = abs(predicted_answer - correct_answer)
                is_correct = diff < 1e-5
            else:
                predicted_answer = None
                is_correct = False
            
            if is_correct:
                correct += 1
            total += 1
            
            results.append({
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'response': response,
                'correct': is_correct
            })

            if (i + 1) % 5 == 0:
                current_acc = correct/total if total > 0 else 0
                print(f"Progress: [{i+1}/{num_samples}]")
                print(f"Current Acc.: [{current_acc:.2%}]")

    return results, correct/total if total > 0 else 0

In [8]:
def save_final_result(results: List[Dict[str, Any]], accuracy: float, filename: str) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += f"[Details]\n"
    
    for idx, result in enumerate(results):
        result_str += f"Question {idx+1}: {result['question']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)

#### Direct prompting with few-shot example

In [9]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.\n"

    for i in range(num_examples):
        cur_question = train_dataset['question'][i]
        cur_answer = train_dataset['answer'][i].split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [10]:
### 어떤 방식으로 저장되는지 확인해보세요!
PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")

 50%|█████     | 5/10 [00:02<00:02,  1.78it/s]

Progress: [5/10]
Current Acc.: [80.00%]


100%|██████████| 10/10 [00:04<00:00,  2.13it/s]

Progress: [10/10]
Current Acc.: [60.00%]


In [11]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!
### 어떤 방식으로 저장되는지 확인해보세요!

shot = [0, 3, 5]
for i in shot:
    PROMPT = construct_direct_prompt(i)
    VERBOSE = False

    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=PROMPT,
        VERBOSE=VERBOSE,
        num_samples=50
    )
    save_final_result(results, accuracy, f"direct_prompting_{i}.txt")

 10%|█         | 5/50 [00:01<00:18,  2.49it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:04<00:28,  1.39it/s]

Progress: [10/50]
Current Acc.: [70.00%]


 30%|███       | 15/50 [00:15<01:11,  2.04s/it]

...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [03:58<00:35,  7.04s/it]

Progress: [45/50]
Current Acc.: [73.33%]


100%|██████████| 50/50 [04:23<00:00,  5.27s/it]

Progress: [50/50]
Current Acc.: [74.00%]


### Chain-of-Thought prompting with few-shot example
```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 되겠죠?

In [12]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )
    prompt = "Solve the following mathematical problems step by step.\n\n"

    for i in range(num_examples):
        idx = sampled_indices[i]
        question = train_dataset['question'][idx]
        answer = train_dataset['answer'][idx]
        clean_answer = re.sub(r'<<.*?>>', '', answer)
        prompt += f"Question:\n{question}\n\nAnswer:\n{clean_answer}\n\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [13]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!
shot = [0, 3, 5]
for i in shot:
    PROMPT = construct_CoT_prompt(i)
    VERBOSE = False

    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=PROMPT,
        VERBOSE=VERBOSE,
        num_samples=50
    )
    save_final_result(results, accuracy, f"COT_prompting_{i}.txt")

 10%|█         | 5/50 [00:02<00:24,  1.87it/s]

Progress: [5/50]
Current Acc.: [60.00%]


 20%|██        | 10/50 [00:05<00:20,  1.98it/s]

Progress: [10/50]
Current Acc.: [50.00%]


 30%|███       | 15/50 [00:08<00:23,  1.50it/s]

...

Progress: [40/50]
Current Acc.: [65.00%]


 90%|█████████ | 45/50 [07:51<00:52, 10.40s/it]

Progress: [45/50]
Current Acc.: [62.22%]


100%|██████████| 50/50 [08:43<00:00, 10.47s/it]

Progress: [50/50]
Current Acc.: [66.00%]


### Construct your prompt!!

목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올려보기!
- gsm8k의 train 데이터셋에서 예시를 가져온 다음 (자유롭게!)
- 그 예시들에 대한 풀이 과정을 만들어주세요!
- 모든 것들이 자유입니다! Direct Prompting, CoT Prompting을 한 결과보다 정답률만 높으면 돼요.

In [14]:
def construct_my_prompt(num_examples: int = 5) -> str:
    examples = [
        (
            "Janet's ducks lay 16 eggs per day. She eats 3 and bakes with 4. She sells the rest for $2 each. How much does she make?",
            "Janet eats 3 and bakes 4, so she uses 3 + 4 = <<3+4=7>>7 eggs. The remainder is 16 - 7 = <<16-7=9>>9 eggs. She sells them for $2 each, so 9 * 2 = $<<9*2=18>>18.",
            "18"
        ),
        (
            "A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts total?",
            "White fiber is half of blue, so 2 / 2 = <<2/2=1>>1 bolt. Total is 2 + 1 = <<2+1=3>>3 bolts.",
            "3"
        ),
        (
            "James runs 3 sprints 3 times a week. Each sprint is 60 meters. How many meters per week?",
            "He runs 3 sprints * 60 meters = <<3*60=180>>180 meters per day. He runs 3 times a week, so 180 * 3 = <<180*3=540>>540 meters.",
            "540"
        ),
        (
            "John takes care of 10 dogs. Each dog takes 0.5 hours a day. How many hours per week?",
            "Daily time is 10 dogs * 0.5 hours = <<10*0.5=5>>5 hours. Weekly time is 5 hours * 7 days = <<5*7=35>>35 hours.",
            "35"
        ),
        (
            "A candle melts 2 cm every hour. How much shorter after burning from 1PM to 5PM?",
            "The time duration is 5PM - 1PM = <<5-1=4>>4 hours. It melts 2 cm/hour, so 4 * 2 = <<4*2=8>>8 cm.",
            "8"
        )
    ]

    prompt = (
        "Solve the math problem step by step. "
        "The last line must be exactly 'Answer: [number]'\n\n"
    )

    for i in range(min(num_examples, len(examples))):
        q, reasoning, a = examples[i]
        prompt += f"Question:\n{q}\n"
        prompt += f"Reasoning:\n{reasoning}\n" 
        prompt += f"Answer: {a}\n\n"

    prompt += "Question:\n{question}\nReasoning:\nLet's think step by step."
    
    return prompt

In [15]:
# TODO: 만든 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!
shot = [0, 3, 5]
for i in shot:
    PROMPT = construct_my_prompt(i)
    VERBOSE = False

    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=PROMPT,
        VERBOSE=VERBOSE,
        num_samples=50
    )
    save_final_result(results, accuracy, f"My_prompting_{i}.txt")

 10%|█         | 5/50 [00:04<00:56,  1.26s/it]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [00:17<01:38,  2.47s/it]

Progress: [10/50]
Current Acc.: [80.00%]


 30%|███       | 15/50 [00:31<01:32,  2.65s/it]

...

Progress: [40/50]
Current Acc.: [77.50%]


 90%|█████████ | 45/50 [05:12<00:34,  6.85s/it]

Progress: [45/50]
Current Acc.: [77.78%]


100%|██████████| 50/50 [05:46<00:00,  6.92s/it]

Progress: [50/50]
Current Acc.: [80.00%]


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot, 5 shot 정답률을 표로 보여주세요!
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요!
3. 본인이 작성한 프롬프트 기법이 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요!
4. 최종적으로, `PROMPTING.md`에 보고서를 작성해주세요!